## 0. Install Dependencies

Run this cell first, then **restart the kernel**, then run all remaining cells.

In [3]:
%%writefile requirements.txt
torch>=2.5.1
transformers==4.57.6
docling==2.96.0
sentence-transformers==3.4.1
chromadb==0.6.3
rank-bm25==0.2.2
openai==1.84.0
tenacity==9.1.4
pymupdf==1.27.2.3
tqdm==4.67.3
pdfplumber==0.11.4


Writing requirements.txt


In [ ]:
# docling==2.96.0
# sentence-transformers==5.1.0
# chromadb==0.6.3
# rank-bm25==0.2.2
# openai>=1.84.0
# groq>=0.31.0
# tenacity>=9.1.2
# pymupdf>=1.26.0
# tqdm>=4.67.0
# pdfplumber>=0.11.0

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("Installation done.")


# Payer Policy PA Parameter Extraction Pipeline

End-to-end RAG pipeline that extracts **12 Prior Authorization parameters** per brand from PsO payer policy PDFs.  
Uses a dual-model Groq setup — 8B for brand detection, 70B for parameter extraction — with hybrid BM25 + dense retrieval, RRF fusion, and cross-encoder reranking.

| Step | Description |
|------|-------------|
| 0 | Install Dependencies |
| 1 | Configuration and Imports |
| 2 | LLM Client (Groq · 8B + 70B · RateLimiter · in-memory cache) |
| 3 | PDF to Markdown (PyMuPDF + Docling, GPU-safe) |
| 4 | Chunking and Vector Store (700/100 · BM25 + BGE-384 + reranker) |
| 5 | Brand Detection (8B · hybrid search · anchor chunk IDs) |
| 6 | Parameter Extraction and Access Score (70B · two-tier retrieval) |
| 7 | Run Pipeline |
| 8 | Execute |


## 1. Configuration and Imports

In [13]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("OPENROUTER_API_KEY")

In [14]:
# torch must be imported and fully initialised BEFORE docling/transformers
# to prevent "AttributeError: module torch has no attribute _utils"
import torch
import torch._utils
import torch.utils

import csv
import hashlib
import json
import logging
import os
import re
import shutil
import tempfile
import time
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from tqdm.auto import tqdm
import fitz
import chromadb
from chromadb.config import Settings as ChromaSettings
import numpy as np
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tenacity import (
    retry, retry_if_exception_type, stop_after_attempt, wait_exponential,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
logging.getLogger("chromadb.telemetry").setLevel(logging.CRITICAL)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Kaggle: Add-ons > Secrets > add OPENROUTER_API_KEY, enable "Attach to session"
# Local:  export OPENROUTER_API_KEY=sk-or-...
OPENROUTER_API_KEY = secret_value_0
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY in Kaggle Secrets (or export OPENROUTER_API_KEY=...)"
print(f"OpenRouter key set: {bool(OPENROUTER_API_KEY)}")

_HERE        = Path(".").resolve()
PDF_DIR = Path("/kaggle/input/datasets/saurav130/document-pdfs/Sample_PsO_ADS_Track")
MARKDOWN_DIR = _HERE / "markdown_output"
OUTPUT_CSV   = _HERE / "submission.csv"
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE = _HERE / "model_cache"
MODEL_CACHE.mkdir(exist_ok=True)

# --- Embeddings: 384-dim bge-small (faster + lighter than bge-base) --------
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
RERANK_MODEL  = "BAAI/bge-reranker-v2-m3"
COLLECTION    = "payer_policy"
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 150
RRF_K         = 60
PARAM_TOP_K   = 4   # max chunks passed to 70B (keeps each call < ~2500 tokens)
HEADER_RATIO  = 0.07
FOOTER_RATIO  = 0.07

# --- OpenRouter models -------------------------------------------------------
OPENROUTER_MODEL_8B  = "meta-llama/llama-3.1-8b-instruct"   # brand detection
OPENROUTER_MODEL_70B = "meta-llama/llama-3.3-70b-instruct"  # parameter extraction
OPENROUTER_BASE_URL  = "https://openrouter.ai/api/v1"

PARAMS = [
    "Age",
    "Step Therapy Requirements Documented in Policy",
    "Number of Steps through Brands",
    "Number of Steps through Generic",
    "Step through-Phototherapy",
    "TB Test required",
    "Initial Authorization Duration(in-months)",
    "Reauthorization Duration(in-months)",
    "Reauthorization Required",
    "Reauthorization Requirements Documented in Policy",
    "Specialist Types",
    "Quantity Limits",
]
CSV_COLUMNS = ["filename", "brand"] + PARAMS + ["access_score"]

print(f"PDF dir      : {PDF_DIR}")
print(f"Output CSV   : {OUTPUT_CSV}")
print(f"Chunk size   : {CHUNK_SIZE}  overlap={CHUNK_OVERLAP}")
print(f"Embed model  : {EMBED_MODEL}  (384-dim)")
print(f"8B model     : {OPENROUTER_MODEL_8B}")
print(f"70B model    : {OPENROUTER_MODEL_70B}")


Device: cuda
OpenRouter key set: True
PDF dir      : /kaggle/input/datasets/saurav130/document-pdfs/Sample_PsO_ADS_Track
Output CSV   : /kaggle/working/submission.csv
Chunk size   : 800  overlap=150
Embed model  : BAAI/bge-small-en-v1.5  (384-dim)
8B model     : meta-llama/llama-3.1-8b-instruct
70B model    : meta-llama/llama-3.3-70b-instruct


## 2. LLM Client

Groq-only, **two-model** design:

| Role | Model | Groq limit |
|------|-------|------------|
| Brand detection | `llama-3.1-8b-instant` | 131K TPM |
| Param extraction | `llama-3.3-70b-versatile` | 12K TPM |

`RateLimiter` tracks requests and tokens in a 60-second sliding window,
staying 1 request and 5% tokens below the free-tier limits.
In-memory cache (MD5 key) avoids duplicate API calls.


In [15]:
# ---------------------------------------------------------------------------
# Module-level response cache  {md5_key -> response_text}
# ---------------------------------------------------------------------------
_llm_cache: Dict[str, str] = {}


def _cache_key(messages: List[Dict], model: str) -> str:
    payload = json.dumps({"m": messages, "model": model}, sort_keys=True)
    return hashlib.md5(payload.encode()).hexdigest()


# ---------------------------------------------------------------------------
# LLM Client — OpenRouter (meta-llama/llama-3.1-8b-instruct or llama-3.3-70b-instruct)
# ---------------------------------------------------------------------------
class LLMClient:
    """
    OpenRouter client wrapping the OpenAI SDK.

    Usage:
        llm_8b  = LLMClient("8b")
        llm_70b = LLMClient("70b")
        text    = llm_8b.complete([{"role": "user", "content": "hi"}])
    """

    def __init__(self, model_size: str = "8b"):
        if model_size not in ("8b", "70b"):
            raise ValueError(f"model_size must be '8b' or '70b', got '{model_size}'")
        self.model_size = model_size
        self.model      = OPENROUTER_MODEL_8B if model_size == "8b" else OPENROUTER_MODEL_70B
        self._client    = OpenAI(
            api_key=OPENROUTER_API_KEY,
            base_url=OPENROUTER_BASE_URL,
        )
        log.info("LLMClient ready — provider=openrouter  model=%s", self.model)

    def complete(
        self,
        messages:      List[Dict],
        temperature:   float = 0.0,
        max_tokens:    int   = 1024,
        use_llm_cache: bool  = True,
    ) -> str:
        cache_key = _cache_key(messages, self.model)
        if use_llm_cache and cache_key in _llm_cache:
            log.debug("Cache hit [%s]", cache_key[:8])
            return _llm_cache[cache_key]

        log.debug("Cache miss [%s] — calling OpenRouter", cache_key[:8])
        response = self._call(messages, temperature, max_tokens)

        if use_llm_cache:
            _llm_cache[cache_key] = response
        return response

    @retry(
        stop=stop_after_attempt(4),
        wait=wait_exponential(multiplier=2, min=4, max=60),
        retry=retry_if_exception_type(Exception),
        reraise=True,
    )
    def _call(self, messages: List[Dict], temperature: float, max_tokens: int) -> str:
        resp = self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content.strip()

    @property
    def cache_size(self) -> int:
        return len(_llm_cache)

    def clear_cache(self) -> None:
        _llm_cache.clear()
        log.info("LLM cache cleared")


print("LLMClient defined (OpenRouter).")


LLMClient defined (OpenRouter).


## 3. PDF to Markdown

PyMuPDF cleans the PDF in memory (headers/footers/links/credentials), then Docling converts it to Markdown with table detection (forced CPU).

In [16]:
HEADER_RATIO = 0.07
FOOTER_RATIO = 0.07

_CRED_PATTERNS = [
    re.compile(r"(username|user[\s_-]*name|login|user[\s_-]*id)\s*[:=]\s*\S+", re.I),
    re.compile(r"(password|passwd|pwd)\s*[:=]\s*\S+", re.I),
    re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"),
]


def _redact_zone(page: fitz.Page, zone: fitz.Rect) -> None:
    for b in page.get_text("blocks", clip=zone):
        page.add_redact_annot(fitz.Rect(b[:4]), fill=(1, 1, 1))


def _redact_credentials(page: fitz.Page) -> None:
    for block in page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE).get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                if any(p.search(span.get("text", "")) for p in _CRED_PATTERNS):
                    page.add_redact_annot(fitz.Rect(span["bbox"]), fill=(1, 1, 1))



_REF_PATTERNS = [
    re.compile(r'^\s*references\s*$', re.I),
    re.compile(r'the above policy is based on the following references', re.I),
]


def _strip_references(doc: fitz.Document) -> None:
    """
    Find the first page containing a References heading and redact everything
    from that heading to the end of the document.
    Pages after the references page are deleted entirely.
    """
    ref_page_idx = None
    ref_y = None

    for page_num in range(len(doc)):
        page = doc[page_num]
        blocks = page.get_text("blocks")
        for b in blocks:
            text = b[4].strip()
            if any(p.search(text) for p in _REF_PATTERNS):
                ref_page_idx = page_num
                ref_y = b[1]  # top y-coordinate of the references block
                break
        if ref_page_idx is not None:
            break

    if ref_page_idx is None:
        return

    # Redact from ref_y to bottom of the references page
    ref_page = doc[ref_page_idx]
    h = ref_page.rect.height
    w = ref_page.rect.width
    ref_page.add_redact_annot(fitz.Rect(0, ref_y, w, h), fill=(1, 1, 1))
    ref_page.apply_redactions()

    # Delete all pages after the references page
    for page_num in range(len(doc) - 1, ref_page_idx, -1):
        doc.delete_page(page_num)


def _clean_doc(doc: fitz.Document) -> None:
    for page in doc:
        h, w = page.rect.height, page.rect.width
        _redact_zone(page, fitz.Rect(0, 0, w, h * HEADER_RATIO))
        _redact_zone(page, fitz.Rect(0, h * (1 - FOOTER_RATIO), w, h))
        for link in page.get_links():
            page.delete_link(link)
        _redact_credentials(page)
        page.apply_redactions()


def _hide_cuda():
    """Return current CUDA_VISIBLE_DEVICES value and set it to empty string."""
    bak = os.environ.get("CUDA_VISIBLE_DEVICES")
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    return bak


def _restore_cuda(bak: Optional[str]) -> None:
    """Restore CUDA_VISIBLE_DEVICES to its original value."""
    if bak is None:
        os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    else:
        os.environ["CUDA_VISIBLE_DEVICES"] = bak




def build_converter() -> DocumentConverter:
    """
    Build the Docling DocumentConverter with CUDA hidden.
    This prevents RT-DETRv2 from attempting to load the MultiScaleDeformableAttention
    CUDA kernel (not compiled on Kaggle), which would otherwise print the
    "Could not load custom kernel" warning dozens of times.
    CUDA is restored after the converter is built so embeddings still use GPU.
    """
    bak = _hide_cuda()
    try:
        opts = PdfPipelineOptions()
        opts.do_ocr = False
        opts.do_table_structure = True
        opts.table_structure_options.do_cell_matching = True

        try:
            from docling.datamodel.pipeline_options import AcceleratorOptions
            from docling.datamodel.accelerator_options import AcceleratorDevice
            opts.accelerator_options = AcceleratorOptions(
                num_threads=4, device=AcceleratorDevice.CPU
            )
        except ImportError:
            pass

        return DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
        )
    finally:
        _restore_cuda(bak)


# ---------------------------------------------------------------------------
# pdfplumber — tabular PDF detection and extraction
# ---------------------------------------------------------------------------
def _is_tabular_pdf(pdf_path: Path, threshold: float = 0.5) -> bool:
    """
    Return True if page 1 is dominated by tables (table area > threshold of page area).
    Used to route tabular PDFs (like PA Detail sheets) away from Docling.
    """
    try:
        import pdfplumber
        with pdfplumber.open(str(pdf_path)) as pdf:
            if not pdf.pages:
                return False
            page      = pdf.pages[0]
            page_area = page.width * page.height
            if page_area == 0:
                return False
            table_area = sum(
                (t.bbox[2] - t.bbox[0]) * (t.bbox[3] - t.bbox[1])
                for t in page.find_tables()
            )
            ratio = table_area / page_area
            log.info("[detect]  %s — table area ratio: %.2f", pdf_path.name, ratio)
            return ratio > threshold
    except Exception as e:
        log.warning("[detect]  %s — tabular check failed: %s", pdf_path.name, e)
        return False


def convert_pdf_tabular(pdf_path: Path, md_dir: Path) -> Optional[Path]:
    """
    Extract tabular PDFs (e.g. PA Detail sheets) using pdfplumber.

    Each drug row is converted to a labeled prose block:
        ## ADALIMUMAB (HUMIRA)
        **Required Medical Information:** Diagnosis. For RA...
        **Age Restriction:** 2 years or older.
        ...

    This format is far easier for the LLM to parse than a distorted wide table.
    """
    import pdfplumber

    md_out = md_dir / (pdf_path.stem + ".md")
    if md_out.exists():
        log.info("[skip]    %s — markdown already exists", pdf_path.name)
        return md_out

    def _clean(cell) -> str:
        if cell is None:
            return ""
        return " ".join(str(cell).split())   # collapse whitespace / newlines

    _HEADER_MARKERS = {"Group", "Indication Indicator", "Required Medical Information"}

    try:
        headers: Optional[List[str]] = None
        rows:    List[List[str]]     = []

        with pdfplumber.open(str(pdf_path)) as pdf:
            for page in pdf.pages:
                for table in page.find_tables():
                    extracted = table.extract()
                    if not extracted:
                        continue
                    for row in extracted:
                        cleaned = [_clean(c) for c in row]

                        # Detect and store header row (first occurrence)
                        if headers is None and _HEADER_MARKERS & set(cleaned):
                            headers = cleaned
                            continue

                        # Skip repeated header rows on subsequent pages
                        if headers and cleaned == headers:
                            continue

                        # Skip blank rows
                        if not any(cleaned):
                            continue

                        rows.append(cleaned)

        if not rows:
            log.warning("[tabular] %s — no rows extracted by pdfplumber", pdf_path.name)
            return None

        # Build labeled prose blocks — one section per drug
        lines = ["# Prior Authorization Detail\n"]
        for row in rows:
            if headers:
                row = (row + [""] * len(headers))[:len(headers)]
            brand = row[0] if row else "Unknown"
            lines.append(f"\n## {brand}\n")
            col_range = enumerate(headers[1:], start=1) if headers else enumerate(row[1:], start=1)
            for i, header in col_range:
                val = row[i] if i < len(row) else ""
                if val:
                    lines.append(f"**{header}:** {val}")

        markdown = "\n".join(lines)
        md_dir.mkdir(parents=True, exist_ok=True)
        md_out.write_text(markdown, encoding="utf-8")
        log.info("[tabular] %s  | drugs: %d | chars: %d",
                 pdf_path.name, len(rows), len(markdown))
        return md_out

    except Exception as exc:
        log.error("[fail]    %s (pdfplumber) — %s", pdf_path.name, exc)
        return None


# ---------------------------------------------------------------------------
# Main converter — routes tabular PDFs to pdfplumber, others to Docling
# ---------------------------------------------------------------------------
def convert_pdf(
    pdf_path:  Path,
    md_dir:    Path,
    converter: DocumentConverter,
) -> Optional[Path]:
    """
    Convert a PDF to Markdown.
    - Tabular PDFs (PA Detail sheets with wide column tables) → pdfplumber
    - Document-style PDFs → Docling
    Skips if the .md file already exists.
    """
    md_out = md_dir / (pdf_path.stem + ".md")
    if md_out.exists():
        log.info("[skip]    %s — markdown already exists", pdf_path.name)
        return md_out

    # Route tabular PDFs to pdfplumber
    if _is_tabular_pdf(pdf_path):
        log.info("[route]   %s — tabular PDF → pdfplumber", pdf_path.name)
        result = convert_pdf_tabular(pdf_path, md_dir)
        if result:
            return result
        log.warning("[route]   %s — pdfplumber failed, falling back to Docling", pdf_path.name)

    # Docling path (document-style PDFs)
    bak      = _hide_cuda()
    tmp_path = None
    try:
        doc = fitz.open(str(pdf_path))
        _clean_doc(doc)
        _strip_references(doc)

        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
            tmp_path = Path(tmp.name)
            doc.save(str(tmp_path), garbage=4, deflate=True)
        doc.close()

        result   = converter.convert(str(tmp_path))
        tmp_path.unlink(missing_ok=True)
        tmp_path = None

        document = result.document
        markdown = document.export_to_markdown()

        n_tables = 0
        try:
            from docling_core.types.doc import DocItemLabel
            n_tables = sum(
                1 for item, _ in document.iterate_items()
                if getattr(item, "label", None) == DocItemLabel.TABLE
            )
        except Exception:
            n_tables = markdown.count("\n|")

        md_dir.mkdir(parents=True, exist_ok=True)
        md_out.write_text(markdown, encoding="utf-8")
        log.info("[done]    %s  | tables: %d | chars: %d",
                 pdf_path.name, n_tables, len(markdown))
        return md_out

    except Exception as exc:
        log.error("[fail]    %s — %s", pdf_path.name, exc)
        if tmp_path:
            Path(tmp_path).unlink(missing_ok=True)
        log.warning("[fallback] %s — Docling failed, retrying with pdfplumber", pdf_path.name)
        return convert_pdf_tabular(pdf_path, md_dir)

    finally:
        _restore_cuda(bak)


print("PDF to Markdown functions defined.")


PDF to Markdown functions defined.


## 4. Chunking and Vector Store

Recursive character splitting (**700 chars / 100 overlap**),
ChromaDB storage, BM25 + 384-dim cosine + RRF + cross-encoder reranker.


In [17]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------



# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------
@dataclass
class Chunk:
    chunk_id: str
    text: str
    columns: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)


# ---------------------------------------------------------------------------
# Recursive text splitter
# ---------------------------------------------------------------------------
_SEPARATORS = ["\n## ", "\n### ", "\n#### ", "\n\n", "\n", ". ", " ", ""]


def _merge_splits(splits: List[str], sep: str, size: int, overlap: int) -> List[str]:
    """
    Merge atomic splits into chunks <= size chars.
    Sliding-window: flush when the next split would exceed size,
    pop from the front until retained context <= overlap.
    """
    chunks: List[str] = []
    window: List[str] = []
    window_len = 0
    sep_len = len(sep)

    def flush() -> None:
        if not window:
            return
        chunk = sep.join(window)
        if len(chunk) <= size:
            chunks.append(chunk)
        else:
            step = max(1, size - overlap)
            for i in range(0, len(chunk), step):
                piece = chunk[i: i + size]
                if piece.strip():
                    chunks.append(piece)

    for s in splits:
        s_len = len(s)
        add_len = s_len + (sep_len if window else 0)

        if window_len + add_len > size:
            flush()
            while window and window_len > overlap:
                removed = window.pop(0)
                window_len -= len(removed) + (sep_len if window else 0)

        window.append(s)
        window_len += s_len + (sep_len if len(window) > 1 else 0)

    flush()
    return chunks


def _recursive_split(text: str, size: int = CHUNK_SIZE,
                     overlap: int = CHUNK_OVERLAP,
                     seps: List[str] = _SEPARATORS) -> List[str]:
    """
    Recursively split text using separators from coarsest to finest,
    then merge into chunks of at most `size` chars with `overlap`.
    """
    if not text.strip():
        return []
    if len(text) <= size:
        return [text.strip()]

    sep = None
    remaining_seps: List[str] = []
    for i, s in enumerate(seps):
        if s == "" or s in text:
            sep = s
            remaining_seps = seps[i + 1:]
            break

    if sep is None:
        return [text.strip()]

    if sep == "":
        step = max(1, size - overlap)
        return [text[i: i + size].strip()
                for i in range(0, len(text), step)
                if text[i: i + size].strip()]

    flat: List[str] = []
    for p in text.split(sep):
        p = p.strip()
        if not p:
            continue
        if len(p) <= size:
            flat.append(p)
        else:
            flat.extend(_recursive_split(p, size, overlap, remaining_seps))

    join_sep = sep.strip() or " "
    return [c.strip() for c in _merge_splits(flat, join_sep, size, overlap) if c.strip()]


# ---------------------------------------------------------------------------
# Table helpers
# ---------------------------------------------------------------------------
_TABLE_RE = re.compile(
    r"(\|[^\n]+\|\n\|[-| :]+\|\n(?:\|[^\n]+\|\n)*)",
    re.MULTILINE,
)
_HEADER_RE = re.compile(r"^#{1,4}\s+(.+)", re.MULTILINE)


def _extract_columns(table_text: str) -> List[str]:
    first_line = table_text.strip().splitlines()[0]
    return [c.strip() for c in first_line.split("|") if c.strip()]


def _split_table(table_text: str, columns: List[str],
                 size: int, meta: Dict) -> List[Chunk]:
    """Split an oversized table into row-batches, each prefixed with the header row."""
    lines = table_text.strip().splitlines()
    if len(lines) < 3:
        return []

    header_block = lines[0] + "\n" + lines[1] + "\n"
    chunks: List[Chunk] = []
    buf = header_block

    for row in lines[2:]:
        candidate = buf + row + "\n"
        if len(candidate) <= size:
            buf = candidate
        else:
            if buf.strip() != header_block.strip():
                chunks.append(Chunk(text=buf.strip(), columns=columns,
                                    metadata=meta, chunk_id=""))
            buf = header_block + row + "\n"

    if buf.strip() and buf.strip() != header_block.strip():
        chunks.append(Chunk(text=buf.strip(), columns=columns,
                            metadata=meta, chunk_id=""))
    return chunks


# ---------------------------------------------------------------------------
# Markdown chunker
# ---------------------------------------------------------------------------
def chunk_markdown(md_text: str, pdf_name: str) -> List[Chunk]:
    """Parse one markdown document into Chunk objects."""
    chunks: List[Chunk] = []
    current_header = "Introduction"
    idx = 0

    def _next_id() -> str:
        nonlocal idx
        cid = f"{Path(pdf_name).stem}_{idx:04d}"
        idx += 1
        return cid

    segments: List[Dict] = []
    last_end = 0
    for m in _TABLE_RE.finditer(md_text):
        if m.start() > last_end:
            segments.append({"type": "prose", "text": md_text[last_end:m.start()]})
        segments.append({"type": "table", "text": m.group(0)})
        last_end = m.end()
    if last_end < len(md_text):
        segments.append({"type": "prose", "text": md_text[last_end:]})

    for seg in segments:
        # ---- TABLE --------------------------------------------------------
        if seg["type"] == "table":
            columns = _extract_columns(seg["text"])
            meta = {"table": True, "pdf": pdf_name, "header": current_header}
            if len(seg["text"]) <= CHUNK_SIZE:
                chunks.append(Chunk(
                    chunk_id=_next_id(),
                    text=seg["text"].strip(),
                    columns=columns,
                    metadata=meta,
                ))
            else:
                for c in _split_table(seg["text"], columns, CHUNK_SIZE, meta):
                    c.chunk_id = _next_id()
                    chunks.append(c)

        # ---- PROSE --------------------------------------------------------
        else:
            prose = seg["text"]
            for hdr in _HEADER_RE.finditer(prose):
                current_header = hdr.group(1).strip()

            for split in _recursive_split(prose):
                for hdr in _HEADER_RE.finditer(split):
                    current_header = hdr.group(1).strip()
                chunks.append(Chunk(
                    chunk_id=_next_id(),
                    text=split,
                    columns=[],
                    metadata={"table": False, "pdf": pdf_name, "header": current_header},
                ))

    return chunks


# ---------------------------------------------------------------------------
# GPU lock — serialises encoder/reranker calls across threads
# (LLM API calls remain fully parallel; only local GPU inference is locked)
# ---------------------------------------------------------------------------
import threading as _threading
_gpu_lock = _threading.Lock()


# ---------------------------------------------------------------------------
# ChromaDB store + hybrid search
# Singleton pattern: models are loaded ONCE in __init__.
# Call init_collection() at the start of each PDF to reset ChromaDB
# without reloading models — prevents the CUDA meta-tensor error that
# occurs when multiple workers each try to load SentenceTransformer/
# CrossEncoder simultaneously onto the GPU.
# ---------------------------------------------------------------------------
class PolicyStore:
    def __init__(self, embed_model: str = EMBED_MODEL, rerank_model: str = RERANK_MODEL):
        log.info("Loading embedding model: %s  (%s, 384-dim)", embed_model, DEVICE)
        self.encoder = SentenceTransformer(
            embed_model,
            device=DEVICE,
            cache_folder=str(MODEL_CACHE),
            model_kwargs={"low_cpu_mem_usage": False},
        )
        log.info("Embedding dim: %d", self.encoder.get_sentence_embedding_dimension())

        log.info("Loading reranker: %s  (%s)", rerank_model, DEVICE)
        self.reranker = CrossEncoder(
            rerank_model,
            device=DEVICE,
            cache_folder=str(MODEL_CACHE),
            model_kwargs={"low_cpu_mem_usage": False},
        )

        self.client: Optional[chromadb.EphemeralClient] = None
        self.col = None
        self._bm25: Optional[BM25Okapi] = None
        self._bm25_ids: List[str] = []
        self._bm25_texts: List[str] = []

    def init_collection(self) -> None:
        """Reset to a fresh in-memory ChromaDB for the next PDF.

        No disk I/O, no model reload — just a new ephemeral collection.
        Must be called once per PDF before add_chunks().
        """
        self.client = chromadb.EphemeralClient()
        self.col = self.client.get_or_create_collection(
            name=COLLECTION,
            metadata={"hnsw:space": "cosine"},
        )
        self._bm25 = None
        self._bm25_ids = []
        self._bm25_texts = []
        log.info("Collection reset (ephemeral, in-memory)")

    def add_chunks(self, chunks: List[Chunk], batch_size: int = 64) -> int:
        existing = set(self.col.get(include=[])["ids"])
        new = [c for c in chunks if c.chunk_id not in existing]
        if not new:
            return 0

        for i in range(0, len(new), batch_size):
            batch = new[i: i + batch_size]
            texts = [c.text for c in batch]
            with _gpu_lock:
                embeddings = self.encoder.encode(
                    texts, batch_size=32, show_progress_bar=False,
                    normalize_embeddings=True,
                ).tolist()
            self.col.add(
                ids=[c.chunk_id for c in batch],
                documents=texts,
                embeddings=embeddings,
                metadatas=[
                    {
                        **c.metadata,
                        "columns": "|".join(c.columns),
                        "table": str(c.metadata.get("table", False)),
                    }
                    for c in batch
                ],
            )

        self._bm25 = None
        log.info("Stored %d new chunks (%d already present)", len(new), len(existing))
        return len(new)

    def _ensure_bm25(self) -> None:
        if self._bm25 is not None:
            return
        result = self.col.get(include=["documents"])
        self._bm25_ids = result["ids"]
        self._bm25_texts = result["documents"]
        self._bm25 = BM25Okapi([t.lower().split() for t in self._bm25_texts])
        log.info("BM25 index built over %d docs", len(self._bm25_ids))

    def hybrid_search(self, query: str, top_k: int = 10,
                      rerank_candidates: int = 30) -> List[Dict[str, Any]]:
        """
        Three-stage retrieval:
          1. BM25 + dense cosine → RRF → top `rerank_candidates`
          2. bge-reranker-v2-m3 cross-encoder → re-scores each candidate
          3. Return top `top_k` by reranker score
        """
        self._ensure_bm25()
        n_candidates = min(rerank_candidates, max(self.col.count(), 1))

        # --- Stage 1a: BM25 sparse ---
        bm25_scores = self._bm25.get_scores(query.lower().split())
        sparse_ranks = np.argsort(bm25_scores)[::-1][:n_candidates].tolist()

        # --- Stage 1b: dense cosine (GPU-safe) ---
        with _gpu_lock:
            q_vec = self.encoder.encode([query], normalize_embeddings=True).tolist()
        dense = self.col.query(
            query_embeddings=q_vec,
            n_results=n_candidates,
            include=["documents", "metadatas", "distances"],
        )
        dense_ids: List[str] = dense["ids"][0]

        # --- Stage 1c: RRF fusion ---
        rrf: Dict[str, float] = {}
        for rank, arr_idx in enumerate(sparse_ranks):
            cid = self._bm25_ids[arr_idx]
            rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (RRF_K + rank + 1)
        for rank, cid in enumerate(dense_ids):
            rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (RRF_K + rank + 1)

        candidate_ids = sorted(rrf, key=lambda x: rrf[x], reverse=True)[:n_candidates]

        # Fetch full text for all candidates
        fetched = self.col.get(ids=candidate_ids, include=["documents", "metadatas"])
        id_map = {
            cid: (doc, meta)
            for cid, doc, meta in zip(
                fetched["ids"], fetched["documents"], fetched["metadatas"]
            )
        }

        # --- Stage 2: rerank (GPU-safe) ---
        pairs = [(query, id_map[cid][0]) for cid in candidate_ids if cid in id_map]
        with _gpu_lock:
            rerank_scores = self.reranker.predict(pairs)   # shape: (n_candidates,)

        ranked = sorted(
            zip(candidate_ids, rerank_scores),
            key=lambda x: x[1],
            reverse=True,
        )[:top_k]

        # --- Stage 3: build results ---
        return [
            {
                "chunk_id": cid,
                "text": id_map[cid][0],
                "columns": [c for c in id_map[cid][1].get("columns", "").split("|") if c],
                "metadata": {
                    "table":  id_map[cid][1].get("table") == "True",
                    "pdf":    id_map[cid][1].get("pdf", ""),
                    "header": id_map[cid][1].get("header", ""),
                },
                "rrf_score":    round(rrf.get(cid, 0.0), 6),
                "rerank_score": round(float(score), 4),
            }
            for cid, score in ranked if cid in id_map
        ]

print("Chunking and PolicyStore defined.")

Chunking and PolicyStore defined.


## 5. Brand Detection  (8B)

Robust brand detection for PsO policies:

1. Gather high-signal chunks from drug lists, indication sections, criteria sections, and direct raw-text snippets.
2. Ask the LLM for strict JSON with only explicitly listed PsO-relevant products.
3. Normalize, deduplicate, and attach anchor chunk IDs using both direct brand text matches and hybrid search.

Falls back to raw markdown context when retrieval misses the drug-list section.


In [ ]:
MAX_CHARS = 12000
_CONTEXT_TOP_K = 6
_ANCHOR_TOP_K = 8

_DRUG_LIST_QUERIES = [
    "applicable drug list covered drugs product list biologics biosimilars plaque psoriasis",
    "preferred non-preferred formulary brand names plaque psoriasis PsO biologic agents",
    "drug list medications requiring prior authorization plaque psoriasis psoriasis",
    "coverage criteria prior authorization plaque psoriasis biologics products",
    "step therapy criteria psoriasis biologic brand names",
]

_BRAND_PROMPT = """\
You extract brand/product names from payer prior authorization policies.

Goal: identify every explicitly listed product/brand in this policy that is relevant to plaque psoriasis (PsO) extraction.

Use these rules exactly:
1. Set policy_has_pso to "Yes" only if the policy text contains plaque psoriasis, psoriasis vulgaris, PsO, or psoriasis criteria/coverage language.
2. Extract product/brand names from applicable drug lists, covered drug lists, prior authorization drug lists, tables, and PsO criteria sections.
3. Include products that are listed as examples or covered products for PsO, including biosimilars and newly named products.
4. Exclude products that appear only for unrelated indications when the text clearly separates them from PsO.
5. Do not invent brands. If a name is not explicitly present in the policy text, do not include it.
6. Return the brand/product in CAPITAL LETTERS.
7. Strip generic/chemical names and parenthetical text.
   Examples:
   - "Tremfya (guselkumab)" -> "TREMFYA"
   - "adalimumab-adaz (HYRIMOZ)" -> "HYRIMOZ"
8. Preserve multi-word product names when they are the brand, e.g. "SIMPONI ARIA".
9. preferred_status must be:
   - "Preferred" only when the text explicitly says preferred.
   - "Non-preferred" only when the text explicitly says non-preferred, nonpreferred, not preferred, or step/non-formulary equivalent.
   - "Unspecified" otherwise.
10. Output strict JSON only. No markdown. No commentary.

Return exactly this schema:
{{
  "policy_has_pso": "Yes",
  "brands_relevant_to_pso": [
    {{
      "brand": "BRAND NAME",
      "preferred_status": "Preferred | Non-preferred | Unspecified"
    }}
  ]
}}

If there is no PsO policy language, return:
{{
  "policy_has_pso": "No",
  "brands_relevant_to_pso": []
}}

POLICY TEXT:
{policy_text}"""


_STATUS_VALUES = {"Preferred", "Non-preferred", "Unspecified"}


def _truncate(text: str, max_chars: int = MAX_CHARS) -> str:
    text = text.strip()
    if len(text) <= max_chars:
        return text
    head = int(max_chars * 0.55)
    middle = int(max_chars * 0.25)
    tail = max_chars - head - middle
    mid_start = max(0, (len(text) // 2) - (middle // 2))
    return (
        text[:head]
        + "\n\n[...middle of document...]\n\n"
        + text[mid_start: mid_start + middle]
        + "\n\n[...end of document...]\n\n"
        + text[-tail:]
    )


def _clean_brand_name(name: str) -> str:
    """Normalize brand/product names for CSV output and de-duplication."""
    name = re.sub(r"\s*\(.*?\)", "", str(name))
    name = re.sub(r"\b(generic|brand|preferred|non[- ]?preferred)\b", "", name, flags=re.I)
    name = re.sub(r"[^A-Za-z0-9+\-/ ]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip(" -/")
    return name.upper()


def _json_from_text(raw: str) -> Dict[str, Any]:
    start = raw.index("{")
    end = raw.rindex("}") + 1
    return json.loads(raw[start:end])


def _parse_json(raw: str, pdf_name: str) -> Dict[str, Any]:
    try:
        result = _json_from_text(raw)
    except (ValueError, json.JSONDecodeError):
        log.error("Brand JSON parse failed for %s:\n%s", pdf_name, raw[:500])
        return {"policy_has_pso": "No", "brands_relevant_to_pso": []}

    has_pso = str(result.get("policy_has_pso", "No")).strip().title()
    if has_pso not in ("Yes", "No"):
        has_pso = "No"

    cleaned: List[Dict[str, str]] = []
    seen: set = set()
    for item in result.get("brands_relevant_to_pso", []) or []:
        if not isinstance(item, dict):
            continue
        brand = _clean_brand_name(item.get("brand", ""))
        if not brand or brand in seen:
            continue
        status = str(item.get("preferred_status", "Unspecified")).strip()
        if status.lower() in {"nonpreferred", "non preferred", "not preferred"}:
            status = "Non-preferred"
        elif status.lower() == "preferred":
            status = "Preferred"
        elif status not in _STATUS_VALUES:
            status = "Unspecified"
        cleaned.append({"brand": brand, "preferred_status": status})
        seen.add(brand)

    return {
        "policy_has_pso": has_pso,
        "brands_relevant_to_pso": cleaned if has_pso == "Yes" else [],
    }


def _chunk_contains(text: str, *needles: str) -> bool:
    hay = re.sub(r"[^A-Z0-9]+", " ", text.upper())
    return any(re.sub(r"[^A-Z0-9]+", " ", n.upper()).strip() in hay for n in needles if n)


def _pdf_chunks(store: "PolicyStore", pdf_name: str) -> List[Tuple[str, str, Dict[str, Any]]]:
    """Return all chunks for the current PDF from the active Chroma collection."""
    fetched = store.col.get(include=["documents", "metadatas"])
    rows: List[Tuple[str, str, Dict[str, Any]]] = []
    for cid, doc, meta in zip(fetched["ids"], fetched["documents"], fetched["metadatas"]):
        if meta.get("pdf") == pdf_name:
            rows.append((cid, doc, meta))
    return rows


def _collect_context(md_path: Path, store: "PolicyStore", pdf_name: str) -> Tuple[str, List[Dict[str, Any]]]:
    """Build high-recall context for the LLM from retrieval plus raw markdown snippets."""
    seen: set = set()
    context_chunks: List[Dict[str, Any]] = []

    for query in _DRUG_LIST_QUERIES:
        for r in store.hybrid_search(query, top_k=_CONTEXT_TOP_K, rerank_candidates=40):
            if r["metadata"].get("pdf") == pdf_name and r["chunk_id"] not in seen:
                seen.add(r["chunk_id"])
                context_chunks.append(r)

    # Directly add chunks that look like drug lists or PsO criteria; this catches
    # tables/sections that BM25+dense retrieval sometimes ranks too low.
    for cid, doc, meta in _pdf_chunks(store, pdf_name):
        if cid in seen:
            continue
        if _chunk_contains(
            doc,
            "applicable drug", "covered drug", "drug list", "preferred", "non-preferred",
            "nonpreferred", "plaque psoriasis", "psoriasis", "psO", "prior authorization",
        ):
            context_chunks.append({"chunk_id": cid, "text": doc, "metadata": meta})
            seen.add(cid)

    chunk_text = "\n\n--- CHUNK ---\n\n".join(r["text"] for r in context_chunks)
    raw_text = md_path.read_text(encoding="utf-8")
    policy_text = _truncate(chunk_text + "\n\n--- RAW MARKDOWN FALLBACK ---\n\n" + _truncate(raw_text))
    return policy_text, context_chunks


def _anchor_ids_for_brand(store: "PolicyStore", pdf_name: str, brand_name: str) -> List[str]:
    """Find anchor chunks with direct brand matches first, then hybrid search."""
    brand_norm = re.sub(r"[^A-Z0-9]+", " ", brand_name.upper()).strip()
    anchors: List[str] = []
    seen: set = set()

    scored = []
    for cid, doc, _meta in _pdf_chunks(store, pdf_name):
        doc_norm = re.sub(r"[^A-Z0-9]+", " ", doc.upper())
        if brand_norm and brand_norm in doc_norm:
            score = 100 + (20 if "PSORIASIS" in doc_norm else 0) + (10 if "CRITERIA" in doc_norm else 0)
            scored.append((score, cid))
    for _score, cid in sorted(scored, reverse=True):
        if cid not in seen:
            anchors.append(cid)
            seen.add(cid)
        if len(anchors) >= _ANCHOR_TOP_K:
            return anchors

    query = f"{brand_name} plaque psoriasis psoriasis prior authorization criteria step therapy covered drug"
    for r in store.hybrid_search(query, top_k=_ANCHOR_TOP_K, rerank_candidates=40):
        cid = r["chunk_id"]
        if r["metadata"].get("pdf") == pdf_name and cid not in seen:
            anchors.append(cid)
            seen.add(cid)
        if len(anchors) >= _ANCHOR_TOP_K:
            break
    return anchors


def detect_brands(md_path: Path, store: "PolicyStore", llm: LLMClient) -> Dict[str, Any]:
    """
    Detect PsO-relevant brands and attach anchor chunk IDs.

    The detector intentionally uses high-recall context because missing a brand
    is more damaging than passing a few extra candidates to the 8B model.
    """
    pdf_name = md_path.stem + ".pdf"
    policy_text, context_chunks = _collect_context(md_path, store, pdf_name)

    if context_chunks:
        log.info(
            "Brand detection -- %s  (%d chunks + raw fallback, %d chars)",
            pdf_name, len(context_chunks), len(policy_text),
        )
    else:
        log.warning("No retrieved brand context found -- using raw markdown for %s", pdf_name)
        log.info("Brand detection -- %s  (raw fallback, %d chars)", pdf_name, len(policy_text))

    messages = [{"role": "user", "content": _BRAND_PROMPT.format(policy_text=policy_text)}]
    raw = llm.complete(messages, temperature=0.0, max_tokens=2048)
    result = _parse_json(raw, pdf_name)

    brands = result.get("brands_relevant_to_pso", [])
    for brand in brands:
        brand["anchor_chunk_ids"] = _anchor_ids_for_brand(store, pdf_name, brand["brand"])

    log.info(
        "  policy_has_pso=%s  brands=%d  anchored=%d",
        result.get("policy_has_pso", "?"),
        len(brands),
        sum(1 for b in brands if b.get("anchor_chunk_ids")),
    )
    return result


print("Brand detection defined.")


## 6. Parameter Detection and Access Score  (70B)

One focused 70B call per detected brand. Retrieval now combines anchor chunks, direct brand-name scans, brand-specific hybrid search, common parameter searches, and a final top-policy fallback so detected brands are not silently skipped when semantic retrieval misses them.

Access Score (1-100): weighted sum -- higher = easier access.


In [ ]:
# ---------------------------------------------------------------------------
# Prompt -- one brand at a time
# ---------------------------------------------------------------------------
PARAM_CONTEXT_TOP_K = 8
PARAM_HYBRID_CANDIDATES = 50

_PARAM_PROMPT = """\
You extract structured prior authorization policy parameters from payer policy text.

Task: extract the 12 plaque psoriasis (PsO) parameters for exactly one product/brand using ONLY the policy chunks below.

BRAND TO EXTRACT:
  Name             : {brand_name}
  Preferred status : {preferred_status}

Critical rules:
1. Extract PsO/plaque psoriasis/psoriasis vulgaris criteria only. Ignore Crohn's, UC, PsA, RA, HS, AS, nr-axSpA, uveitis, and other indications unless the same requirement is explicitly universal for all indications/products.
2. If the policy separates initial approval and continuation/renewal, use initial approval text for initial parameters and continuation/renewal text for reauthorization parameters.
3. Universal policy requirements that apply to all listed drugs must be combined with brand-specific requirements using AND logic.
4. If the policy gives OR paths, choose the least restrictive valid PsO path for counts and access scoring fields, but preserve the full relevant step-therapy text in the free-text field.
5. Count only requirements explicitly stated in the chunks. Do not infer from drug class, FDA label, or medical knowledge.
6. A step counts only when the patient must try/fail, have contraindication, intolerance, inadequate response, or documented inability to use the therapy before this brand.
7. Do not count the target brand itself as a prerequisite step.
8. Do not count phototherapy in brand/generic step counts; report it only in Step through-Phototherapy.
9. Quantity Limits means explicit quantity, units, days supply, dose cap, max dose, or dispensing limit. Do not copy normal dosing instructions unless they are framed as a limit.
10. Output strict JSON only. No markdown. No explanation.

Parameter definitions:
- Age: explicit age threshold for PsO eligibility. Use "FDA labelled age" only when the policy explicitly defers age to FDA labeling and gives no number. Use "NA" if not mentioned.
- Step Therapy Requirements Documented in Policy: full relevant PsO step-therapy language for this brand, including universal and brand-specific text. Use "NA" if no step therapy is documented.
- Number of Steps through Brands: number of required branded/biologic/JAK/specialty-product steps before this brand. Use the least restrictive OR path. Use "NA" if no brand step is required.
- Number of Steps through Generic: number of required generic/conventional/topical/systemic non-biologic steps before this brand. Use the least restrictive OR path. Use "NA" if no generic step is required.
- Step through-Phototherapy: "Yes" if phototherapy/light therapy is mandatory; "No" if criteria exist but phototherapy is not mandatory; "N/A" if no PsO criteria exist in the chunks.
- TB Test required: "Y" if TB/tuberculosis screening/testing is required before or during therapy; "N" if explicitly not required; "NA" if not mentioned.
- Initial Authorization Duration(in-months): numeric months for initial approval. Convert weeks/days to months when clear, otherwise use "Unspecified" if approval is required but duration is not stated. Use "NA" if not mentioned.
- Reauthorization Duration(in-months): numeric months for renewal/continuation approval. Same conversion rules as above.
- Reauthorization Required: "Yes" if renewal/continuation/reauthorization criteria or duration is documented; "No" if explicitly not required; "NA" if not mentioned.
- Reauthorization Requirements Documented in Policy: actual PsO continuation/renewal criteria text. Use "NA" if not documented.
- Specialist Types: explicit acceptable specialist/prescriber types. Use "NA" if not stated.
- Quantity Limits: explicit quantity-limit text. Use "NA" if not stated.

Return exactly this JSON schema:
{{
  "brand": "{brand_name}",
  "preferred_status": "{preferred_status}",
  "Age": "",
  "Step Therapy Requirements Documented in Policy": "",
  "Number of Steps through Brands": "",
  "Number of Steps through Generic": "",
  "Step through-Phototherapy": "",
  "TB Test required": "",
  "Initial Authorization Duration(in-months)": "",
  "Reauthorization Duration(in-months)": "",
  "Reauthorization Required": "",
  "Reauthorization Requirements Documented in Policy": "",
  "Specialist Types": "",
  "Quantity Limits": ""
}}

POLICY CHUNKS:
{chunks}"""

# ---------------------------------------------------------------------------
# Retrieval queries for parameter extraction
# ---------------------------------------------------------------------------
_COMMON_QUERIES = [
    "plaque psoriasis prior authorization criteria initial approval step therapy",
    "psoriasis continuation renewal reauthorization duration months approval",
    "tuberculosis TB screening test specialist prescriber dermatologist quantity limit",
    "phototherapy topical systemic therapy conventional therapy biologic failure psoriasis",
]

_PARAMETER_FIELD_DEFAULTS = {
    "Age": "NA",
    "Step Therapy Requirements Documented in Policy": "NA",
    "Number of Steps through Brands": "NA",
    "Number of Steps through Generic": "NA",
    "Step through-Phototherapy": "N/A",
    "TB Test required": "NA",
    "Initial Authorization Duration(in-months)": "NA",
    "Reauthorization Duration(in-months)": "NA",
    "Reauthorization Required": "NA",
    "Reauthorization Requirements Documented in Policy": "NA",
    "Specialist Types": "NA",
    "Quantity Limits": "NA",
}


def _param_norm(text: str) -> str:
    return re.sub(r"[^A-Z0-9]+", " ", str(text).upper()).strip()


def _param_pdf_chunks(store: "PolicyStore", pdf_name: str) -> List[Tuple[str, str, Dict[str, Any]]]:
    fetched = store.col.get(include=["documents", "metadatas"])
    rows: List[Tuple[str, str, Dict[str, Any]]] = []
    for cid, doc, meta in zip(fetched["ids"], fetched["documents"], fetched["metadatas"]):
        if meta.get("pdf") == pdf_name:
            rows.append((cid, doc, meta))
    return rows


def _param_chunk_score(text: str, brand_name: str) -> int:
    hay = _param_norm(text)
    brand = _param_norm(brand_name)
    score = 0
    if brand and brand in hay:
        score += 120
    for term, weight in (
        ("PLAQUE PSORIASIS", 35),
        ("PSORIASIS", 25),
        ("PRIOR AUTHORIZATION", 20),
        ("CRITERIA", 20),
        ("INITIAL", 12),
        ("REAUTHORIZATION", 15),
        ("CONTINUATION", 15),
        ("RENEWAL", 15),
        ("STEP", 15),
        ("PHOTOTHERAPY", 12),
        ("TUBERCULOSIS", 12),
        (" TB ", 12),
        ("QUANTITY", 12),
        ("SPECIALIST", 10),
        ("DERMATOLOGIST", 10),
    ):
        if term in hay:
            score += weight
    return score


def _append_param_chunk(cid: str, doc: str, seen: set, selected: List[Tuple[str, str]]) -> None:
    if cid not in seen and len(selected) < PARAM_CONTEXT_TOP_K:
        seen.add(cid)
        selected.append((cid, doc))


# ---------------------------------------------------------------------------
# Parameter chunk retrieval
# ---------------------------------------------------------------------------
def _get_brand_chunks(
    store:            "PolicyStore",
    pdf_name:         str,
    brand_name:       str,
    anchor_chunk_ids: List[str],
) -> str:
    """
    High-recall retrieval for one brand:
      1. Anchor chunks from brand detection.
      2. Direct brand-name scan over all chunks for this PDF.
      3. Brand-specific hybrid search.
      4. Common parameter searches.
      5. Top policy chunks fallback so detected brands are not skipped.
    """
    seen: set = set()
    selected: List[Tuple[str, str]] = []

    if anchor_chunk_ids:
        try:
            fetched = store.col.get(ids=anchor_chunk_ids, include=["documents", "metadatas"])
            for cid, doc, meta in zip(fetched["ids"], fetched["documents"], fetched["metadatas"]):
                if meta.get("pdf") == pdf_name:
                    _append_param_chunk(cid, doc, seen, selected)
        except Exception as exc:
            log.warning("Anchor fetch failed for %s: %s", brand_name, exc)

    brand_norm = _param_norm(brand_name)
    pdf_chunks = _param_pdf_chunks(store, pdf_name)

    for cid, doc, _meta in sorted(
        pdf_chunks,
        key=lambda row: _param_chunk_score(row[1], brand_name),
        reverse=True,
    ):
        if len(selected) >= PARAM_CONTEXT_TOP_K:
            break
        if brand_norm and brand_norm in _param_norm(doc):
            _append_param_chunk(cid, doc, seen, selected)

    if len(selected) < PARAM_CONTEXT_TOP_K:
        brand_query = (
            f"{brand_name} plaque psoriasis prior authorization criteria initial approval "
            f"step therapy reauthorization continuation TB quantity limit specialist"
        )
        for r in store.hybrid_search(
            brand_query,
            top_k=PARAM_CONTEXT_TOP_K,
            rerank_candidates=PARAM_HYBRID_CANDIDATES,
        ):
            if r["metadata"].get("pdf") == pdf_name:
                _append_param_chunk(r["chunk_id"], r["text"], seen, selected)
            if len(selected) >= PARAM_CONTEXT_TOP_K:
                break

    for query in _COMMON_QUERIES:
        if len(selected) >= PARAM_CONTEXT_TOP_K:
            break
        for r in store.hybrid_search(query, top_k=4, rerank_candidates=PARAM_HYBRID_CANDIDATES):
            if r["metadata"].get("pdf") == pdf_name:
                _append_param_chunk(r["chunk_id"], r["text"], seen, selected)
            if len(selected) >= PARAM_CONTEXT_TOP_K:
                break

    if not selected:
        log.warning("    Retrieval miss for %s -- using top scored PDF chunks as fallback", brand_name)
        for cid, doc, _meta in sorted(
            pdf_chunks,
            key=lambda row: _param_chunk_score(row[1], brand_name),
            reverse=True,
        )[:PARAM_CONTEXT_TOP_K]:
            _append_param_chunk(cid, doc, seen, selected)

    labelled = [f"[chunk_id: {cid}]\n{doc}" for cid, doc in selected[:PARAM_CONTEXT_TOP_K]]
    return "\n\n---\n\n".join(labelled)


# ---------------------------------------------------------------------------
# JSON + scoring helpers
# ---------------------------------------------------------------------------
def _parse_brand_json(raw: str, brand_name: str) -> Dict[str, Any]:
    try:
        start = raw.index("{")
        end = raw.rindex("}") + 1
        result = json.loads(raw[start:end])
    except (ValueError, json.JSONDecodeError):
        log.error("JSON parse failed for brand '%s':\n%s", brand_name, raw[:300])
        return {}

    if not isinstance(result, dict):
        return {}
    result["brand"] = str(result.get("brand") or brand_name).strip().upper()
    for field, default in _PARAMETER_FIELD_DEFAULTS.items():
        value = result.get(field, default)
        if value is None or str(value).strip() == "":
            value = default
        result[field] = str(value).strip()
    return result


def _score_age(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", "", "fda labelled age", "fda labeled age"):
        return 1.0
    m = re.search(r"(\d+)", v)
    if not m:
        return 0.7
    age = int(m.group(1))
    if age <= 6:   return 1.0
    if age <= 12:  return 0.9
    if age <= 18:  return 0.7
    return 0.4


def _score_steps(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", "", "0"):  return 1.0
    m = re.search(r"(\d+)", v)
    if not m:                  return 1.0
    return max(0.0, 1.0 - int(m.group(1)) * 0.3)


def _score_duration(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", "", "unspecified"):  return 0.5
    m = re.search(r"(\d+)", v)
    if not m:  return 0.5
    months = int(m.group(1))
    if months >= 12:  return 1.0
    if months >= 6:   return 0.7
    if months >= 3:   return 0.4
    return 0.2


def _score_yesno(val: str, yes_score: float = 0.3, no_score: float = 1.0) -> float:
    v = val.strip().lower()
    if v in ("yes", "y"):  return yes_score
    if v in ("no", "n"):   return no_score
    return 0.7


def _score_text_present(val: str) -> float:
    v = val.strip().lower()
    return 0.3 if v and v != "na" else 1.0


_WEIGHTS = {
    "Number of Steps through Brands":                   10,
    "Initial Authorization Duration(in-months)":        15,
    "TB Test required":                                 15,
    "Age":                                              20,
    "Number of Steps through Generic":                  10,
    "Step through-Phototherapy":                         5,
    "Step Therapy Requirements Documented in Policy":    5,
    "Reauthorization Required":                          5,
    "Reauthorization Duration(in-months)":               5,
    "Specialist Types":                                  4,
    "Reauthorization Requirements Documented in Policy": 3,
    "Quantity Limits":                                   3,
}


def compute_access_score(row: Dict[str, str]) -> int:
    scorers = {
        "Age":                                            lambda v: _score_age(v),
        "Step Therapy Requirements Documented in Policy": lambda v: _score_text_present(v),
        "Number of Steps through Brands":                 lambda v: _score_steps(v),
        "Number of Steps through Generic":                lambda v: _score_steps(v),
        "Step through-Phototherapy":                      lambda v: _score_yesno(v),
        "TB Test required":                               lambda v: _score_yesno(v, yes_score=0.3, no_score=1.0),
        "Initial Authorization Duration(in-months)":      lambda v: _score_duration(v),
        "Reauthorization Duration(in-months)":            lambda v: _score_duration(v),
        "Reauthorization Required":                       lambda v: _score_yesno(v),
        "Reauthorization Requirements Documented in Policy": lambda v: _score_text_present(v),
        "Specialist Types":                               lambda v: _score_text_present(v),
        "Quantity Limits":                                lambda v: _score_text_present(v),
    }
    return round(sum(_WEIGHTS[p] * scorers[p](row.get(p, "NA")) for p in _WEIGHTS))


def _flatten_row(filename: str, brand_result: Dict) -> Dict[str, str]:
    row = {"filename": filename, "brand": brand_result.get("brand", "")}
    for p in PARAMS:
        row[p] = str(brand_result.get(p, _PARAMETER_FIELD_DEFAULTS.get(p, "NA")))
    row["access_score"] = str(compute_access_score(row))
    return row


# ---------------------------------------------------------------------------
# Core extraction -- one 70B LLM call per brand
# ---------------------------------------------------------------------------
def extract_params(
    md_path:    Path,
    brand_data: Dict[str, Any],
    store:      "PolicyStore",
    llm:        LLMClient,
) -> List[Dict[str, str]]:
    """For each detected brand, retrieve high-recall chunks and extract 12 parameters."""
    pdf_name = md_path.stem + ".pdf"
    brands = brand_data.get("brands_relevant_to_pso", [])
    if not brands:
        log.info("  No brands -- skipping")
        return []

    _done: set = set()
    if OUTPUT_CSV.exists():
        with open(OUTPUT_CSV, newline="", encoding="utf-8") as _f:
            for _r in csv.DictReader(_f):
                _done.add((_r.get("filename", ""), _r.get("brand", "")))

    seen_brands: set = set()
    unique_brands = []
    for b in brands:
        brand_name = str(b.get("brand", "")).strip().upper()
        if brand_name and brand_name not in seen_brands:
            b["brand"] = brand_name
            seen_brands.add(brand_name)
            unique_brands.append(b)
    if len(unique_brands) < len(brands):
        log.info("  Deduplicated brands: %d -> %d", len(brands), len(unique_brands))
    brands = unique_brands

    rows: List[Dict[str, str]] = []
    for brand in tqdm(brands, desc=f"  {pdf_name}", unit="brand"):
        brand_name = brand["brand"]
        preferred = brand.get("preferred_status", "Unspecified")
        anchor_ids = brand.get("anchor_chunk_ids", [])

        if (pdf_name, brand_name) in _done:
            log.info("  Skipping (already in CSV) -- %s", brand_name)
            continue

        log.info("  Extracting -- %s (%s)  anchors=%d", brand_name, preferred, len(anchor_ids))
        chunks = _get_brand_chunks(store, pdf_name, brand_name, anchor_ids)
        if not chunks.strip():
            log.warning("    No chunks retrieved for %s -- skipping", brand_name)
            continue
        log.info("    Chunks: %d chars -> 70B LLM", len(chunks))

        messages = [{
            "role": "user",
            "content": _PARAM_PROMPT.format(
                brand_name=brand_name,
                preferred_status=preferred,
                chunks=chunks,
            ),
        }]
        raw = llm.complete(messages, temperature=0.0, max_tokens=1400)
        result = _parse_brand_json(raw, brand_name)
        if result:
            row = _flatten_row(pdf_name, result)
            rows.append(row)
            write_csv([row], OUTPUT_CSV, append=True)
            _done.add((pdf_name, brand_name))
            log.info("    Saved to CSV: %s", brand_name)

    log.info("  Total rows extracted: %d", len(rows))
    return rows


# ---------------------------------------------------------------------------
# CSV writer
# ---------------------------------------------------------------------------
_csv_lock = __import__("threading").Lock()

def write_csv(rows: List[Dict], output_path: Path, append: bool = True) -> None:
    with _csv_lock:
        existing: set = set()
        if append and output_path.exists():
            with open(output_path, newline="", encoding="utf-8") as f:
                for r in csv.DictReader(f):
                    existing.add((r.get("filename", ""), r.get("brand", "")))

        new_rows = [r for r in rows if (r["filename"], r["brand"]) not in existing]
        skipped = len(rows) - len(new_rows)
        if skipped:
            log.info("  Skipped %d duplicate row(s) already in CSV", skipped)
        if not new_rows:
            log.info("  No new rows to write")
            return

        mode = "a" if (append and output_path.exists()) else "w"
        write_header = not (append and output_path.exists())
        with open(output_path, mode, newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
            if write_header:
                writer.writeheader()
            writer.writerows(new_rows)
        log.info("CSV updated -- %d new row(s) -> %s", len(new_rows), output_path)


print("Parameter detection and access score defined.")


## 7. Run Pipeline

Full end-to-end pipeline for a single PDF:

1. PDF -> Markdown (Docling, CPU)
2. Chunk & Store (700/100, 384-dim bge-small)
3. Brand Detection (8B + hybrid search)
4. Parameter Extraction (70B + two-tier retrieval, rate-limited)
5. Write CSV


In [21]:
def run_pipeline(pdf_path: Path, store: "PolicyStore") -> None:
    """Run the full pipeline for one PDF.

    Args:
        pdf_path: Path to the PDF file.
        store: A pre-built PolicyStore (models loaded once externally).
               init_collection() is called here to reset ChromaDB for this PDF.
    """
    log.info("=" * 65)
    log.info("PDF  : %s", pdf_path.name)
    log.info("=" * 65)

    # ------------------------------------------------------------------
    # STEP 1 -- PDF -> Markdown  (skipped if .md already exists)
    # ------------------------------------------------------------------
    log.info("STEP 1 -- PDF -> Markdown")
    md_path = MARKDOWN_DIR / (pdf_path.stem + ".md")
    if not md_path.exists():
        converter = build_converter()
        md_path   = convert_pdf(pdf_path, MARKDOWN_DIR, converter)
    if md_path is None or not md_path.exists():
        log.error("Markdown conversion failed -- aborting")
        return
    log.info("  Markdown: %s", md_path)

    # ------------------------------------------------------------------
    # STEP 2 -- Chunk & Store  (fresh in-memory collection per PDF)
    # ------------------------------------------------------------------
    log.info("STEP 2 -- Chunk & Store  [bge-small-en-v1.5, 384-dim]")
    store.init_collection()          # reset ChromaDB — no model reload
    pdf_name = pdf_path.stem + ".pdf"
    md_text  = md_path.read_text(encoding="utf-8")
    chunks   = chunk_markdown(md_text, pdf_name)
    log.info("  Chunks produced: %d", len(chunks))
    store.add_chunks(chunks)

    # ------------------------------------------------------------------
    # STEP 3 -- Brand Detection  [8B]
    # ------------------------------------------------------------------
    log.info("STEP 3 -- Brand Detection  [%s]", OPENROUTER_MODEL_8B)
    llm_8b     = LLMClient("8b")
    brand_data = detect_brands(md_path, store, llm_8b)

    has_pso = brand_data.get("policy_has_pso", "No")
    brands  = brand_data.get("brands_relevant_to_pso", [])
    log.info("  policy_has_pso : %s", has_pso)
    log.info("  brands found   : %s", [b["brand"] for b in brands])

    if has_pso != "Yes" or not brands:
        log.info("  No PsO brands -- nothing to extract")
        return

    # ------------------------------------------------------------------
    # STEP 4 -- Parameter Extraction  [70B]
    # ------------------------------------------------------------------
    log.info("STEP 4 -- Parameter Extraction  [%s]", OPENROUTER_MODEL_70B)
    llm_70b = LLMClient("70b")
    rows    = extract_params(md_path, brand_data, store, llm_70b)

    # ------------------------------------------------------------------
    # STEP 5 -- Summary
    # ------------------------------------------------------------------
    if rows:
        log.info("STEP 5 -- Done: %d brand(s) extracted and saved for %s", len(rows), pdf_path.name)
        for r in rows:
            log.info(
                "    %-30s  steps_brand=%-4s  reauth=%-4s  TB=%s  score=%s",
                r["brand"],
                r["Number of Steps through Brands"],
                r["Reauthorization Required"],
                r["TB Test required"],
                r["access_score"],
            )
    else:
        log.info("  No rows extracted for %s", pdf_path.name)

    log.info("LLM cache: %d entries", llm_8b.cache_size)


print("run_pipeline defined.")

run_pipeline defined.


## 8. Execute

Processes every PDF in `PDF_DIR` sequentially. Already-converted markdowns are skipped (Step 1 caches them). Already-extracted `(filename, brand)` pairs are skipped in the CSV.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# Set to None to process all PDFs, or a number like 5 to limit.
MAX_PDFS        = None
# Docling/PyTorch converter setup is not thread-safe on Kaggle because CUDA_VISIBLE_DEVICES
# is process-global. Keep PDF conversion sequential; the GPU models are still loaded once
# and reused during extraction.
CONVERT_WORKERS = 1

pdfs = sorted(PDF_DIR.glob("*.pdf"))
if not pdfs:
    raise FileNotFoundError(f"No PDFs found in {PDF_DIR}")
if MAX_PDFS is not None:
    pdfs = pdfs[:MAX_PDFS]

print(f"Processing {len(pdfs)} PDF(s)  (MAX_PDFS={MAX_PDFS})")

# Partition: markdowns already on disk vs. need conversion
md_ready  = [p for p in pdfs if (MARKDOWN_DIR / (p.stem + ".md")).exists()]
md_needed = [p for p in pdfs if p not in set(md_ready)]
print(f"  Markdowns already done : {len(md_ready)}")
print(f"  Need conversion        : {len(md_needed)}")

# ------------------------------------------------------------------
# Phase 1: convert PDFs that don't have markdowns yet (parallel)
# ------------------------------------------------------------------
if md_needed:
    print(f"\nPhase 1: PDF -> Markdown ({len(md_needed)} PDF(s), {CONVERT_WORKERS} workers)")

    def _convert(pdf):
        conv = build_converter()
        return pdf, convert_pdf(pdf, MARKDOWN_DIR, conv)

    with ThreadPoolExecutor(max_workers=CONVERT_WORKERS) as pool:
        futures = {pool.submit(_convert, pdf): pdf for pdf in md_needed}
        with tqdm(total=len(md_needed), desc="Converting", unit="pdf") as pbar:
            for fut in as_completed(futures):
                _pdf, _md = fut.result()
                if _md is None:
                    log.error("Conversion failed for %s", _pdf.name)
                pbar.update(1)
else:
    print("\nPhase 1: All markdowns already exist — skipping conversion")

# ------------------------------------------------------------------
# Phase 2: brand detection + extraction (sequential)
#
# ONE PolicyStore is created here — models loaded once and shared
# across all PDFs. Each PDF calls store.init_collection() internally
# to reset the in-memory ChromaDB without reloading models.
# ------------------------------------------------------------------
print(f"\nPhase 2: Brand detection + extraction (sequential)")
print("Loading embedding + reranker models (once)...")
_store = PolicyStore()
print("Models loaded.")

all_pdfs = md_ready + md_needed
for i, pdf in enumerate(tqdm(all_pdfs, desc="Extracting", unit="pdf")):
    print(f"\n[{i+1}/{len(all_pdfs)}] {pdf.name}")
    try:
        run_pipeline(pdf, _store)
    except Exception as e:
        log.error("Pipeline failed for %s: %s", pdf.name, e, exc_info=True)

print(f"\nAll done. Output: {OUTPUT_CSV}")